# Activity 13: Evaluation Metrics Engine

## Learning Objectives
By the end of this activity you will be able to:
- Implement Accuracy, Precision, Recall, F1-Score, and AUC-ROC **from scratch** using NumPy only
- Verify each implementation against its `sklearn.metrics` equivalent
- Apply all metrics across **StratifiedKFold** folds and report mean ± standard deviation
- Interpret metric variance across folds as a measure of model stability vs dataset variance
- Choose the right metric for imbalanced vs balanced datasets

---
> **Why implement metrics from scratch?**  
> sklearn gives you a number. Building it yourself lets you understand *when that number lies*.  
> F1 of 0.9 on an imbalanced dataset is very different from F1 of 0.9 on a balanced one.

## Part 1 — Imports & Datasets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix
)

np.random.seed(42)

# --- Balanced dataset ---
X_bal, y_bal = make_classification(
    n_samples=1500, n_features=10, n_informative=8,
    n_redundant=2, random_state=42
)

# --- Imbalanced dataset (10% positive class) ---
# WHY imbalanced?  Real-world fraud/medical/churn data is rarely 50/50.
X_imb, y_imb = make_classification(
    n_samples=1500, n_features=10, n_informative=8,
    n_redundant=2, weights=[0.90, 0.10],   # 90/10 split
    random_state=42
)

scaler = StandardScaler()
X_bal = scaler.fit_transform(X_bal)
X_imb = scaler.fit_transform(X_imb)

print(f"Balanced   – Class counts: {np.bincount(y_bal)}")
print(f"Imbalanced – Class counts: {np.bincount(y_imb)}")

## Part 2 — Confusion Matrix from Scratch

All classification metrics derive from the **confusion matrix**:

|  | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

In [ ]:
def confusion_matrix_scratch(y_true, y_pred):
    """
    Returns (TP, FP, TN, FN) as integers.
    WHY integers not floats? These are counts of samples — discretely defined.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    return TP, FP, TN, FN

# Quick test
y_t = np.array([1, 1, 0, 0, 1, 0])
y_p = np.array([1, 0, 0, 1, 1, 0])
TP, FP, TN, FN = confusion_matrix_scratch(y_t, y_p)
print(f"TP={TP}  FP={FP}  TN={TN}  FN={FN}")

# Verify
cm_sk = confusion_matrix(y_t, y_p)
assert TP == cm_sk[1,1] and FP == cm_sk[0,1] and TN == cm_sk[0,0] and FN == cm_sk[1,0]
print("Confusion matrix sanity check passed ✓")

## Part 3 — Accuracy, Precision, Recall, F1 from Scratch

| Metric | Formula | Best for |
|---|---|---|
| Accuracy | $(TP+TN)/(TP+FP+TN+FN)$ | Balanced datasets |
| Precision | $TP/(TP+FP)$ | Minimise false alarms (spam filter) |
| Recall | $TP/(TP+FN)$ | Minimise missed positives (cancer screen) |
| F1 | $2 \cdot \frac{P \cdot R}{P+R}$ | Imbalanced datasets |
| AUC-ROC | Area under ROC curve | Threshold-independent ranking |

In [ ]:
EPS = 1e-9   # Prevent zero division

def accuracy_scratch(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def precision_scratch(y_true, y_pred):
    """
    Of all samples predicted positive, what fraction ACTUALLY is positive?
    COMMON ERROR: Using precision as primary metric for recall-critical tasks
                  (e.g. medical screening — you'd miss real cases).
    """
    TP, FP, _, _ = confusion_matrix_scratch(y_true, y_pred)
    return TP / (TP + FP + EPS)

def recall_scratch(y_true, y_pred):
    """
    Of all actual positives, what fraction did we catch?
    Also called Sensitivity or True Positive Rate.
    """
    TP, _, _, FN = confusion_matrix_scratch(y_true, y_pred)
    return TP / (TP + FN + EPS)

def f1_scratch(y_true, y_pred):
    """
    Harmonic mean of Precision and Recall.
    WHY harmonic mean, not arithmetic?  Punishes extreme imbalance between P and R.
    DEBUG TIP: If F1 is very low despite high accuracy → likely class imbalance issue.
    """
    p = precision_scratch(y_true, y_pred)
    r = recall_scratch(y_true, y_pred)
    return 2 * p * r / (p + r + EPS)

def roc_auc_scratch(y_true, y_proba):
    """
    Computes AUC-ROC using the trapezoidal rule.
    WHY AUC?  Independent of classification threshold — measures overall
    ranking quality.  Random classifier = 0.5, perfect = 1.0.
    """
    y_true  = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    thresholds = np.unique(np.concatenate([[0], np.sort(y_proba)[::-1], [1]]))
    tprs, fprs = [], []
    actual_pos  = np.sum(y_true == 1)
    actual_neg  = np.sum(y_true == 0)

    for t in thresholds:
        y_pred_t = (y_proba >= t).astype(int)
        TP, FP, TN, FN = confusion_matrix_scratch(y_true, y_pred_t)
        tprs.append(TP / (actual_pos + EPS))
        fprs.append(FP / (actual_neg + EPS))

    fprs, tprs = zip(*sorted(zip(fprs, tprs)))
    auc = np.trapz(tprs, fprs)         # trapezoidal integration
    return float(auc), list(fprs), list(tprs)

print("Metric functions defined ✓")

## Part 4 — Verify Against sklearn

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X_bal, y_bal, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr, y_tr)
y_pred   = clf.predict(X_te)
y_proba  = clf.predict_proba(X_te)[:, 1]

metrics_scratch = {
    'Accuracy' : accuracy_scratch(y_te,  y_pred),
    'Precision': precision_scratch(y_te, y_pred),
    'Recall'   : recall_scratch(y_te,    y_pred),
    'F1'       : f1_scratch(y_te,        y_pred),
    'AUC-ROC'  : roc_auc_scratch(y_te,  y_proba)[0],
}
metrics_sklearn = {
    'Accuracy' : accuracy_score(y_te,            y_pred),
    'Precision': precision_score(y_te,           y_pred),
    'Recall'   : recall_score(y_te,              y_pred),
    'F1'       : f1_score(y_te,                  y_pred),
    'AUC-ROC'  : roc_auc_score(y_te,            y_proba),
}

comparison = pd.DataFrame({'Scratch': metrics_scratch, 'sklearn': metrics_sklearn})
comparison['Match'] = (comparison['Scratch'] - comparison['sklearn']).abs() < 0.005
print(comparison.round(4))
assert comparison['Match'].all(), "Metric mismatch!"
print("\nAll metrics match sklearn ✓")

## Part 5 — ROC Curve Comparison

In [ ]:
auc_s, fprs_s, tprs_s = roc_auc_scratch(y_te, y_proba)
fpr_sk, tpr_sk, _ = roc_curve(y_te, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fprs_s, tprs_s, color='steelblue',  lw=2.5, label=f'From Scratch  (AUC={auc_s:.3f})')
plt.plot(fpr_sk, tpr_sk, color='crimson',    lw=1.5, linestyle='--', label=f'sklearn       (AUC={roc_auc_score(y_te, y_proba):.3f})')
plt.plot([0,1],[0,1],    color='gray',       linestyle=':',  label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve: From Scratch vs sklearn')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 6 — K-Fold Metric Evaluation

**Why report metrics across folds?**  
A single train/test split can be lucky or unlucky. K-Fold gives you mean ± std — a measure of *how reliable* the model is, not just how good it appears once.

In [ ]:
def evaluate_kfold(X, y, n_splits=5, label='Dataset'):
    """
    Runs StratifiedKFold and collects all metrics across folds.
    WHY Stratified? Imbalanced classes must be proportionally represented
                    in every fold to get reliable metric estimates.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_metrics = {m: [] for m in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']}

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_v = X[train_idx], X[val_idx]
        y_tr, y_v = y[train_idx], y[val_idx]

        clf = LogisticRegression(max_iter=1000, random_state=42)
        clf.fit(X_tr, y_tr)
        y_pred  = clf.predict(X_v)
        y_prob  = clf.predict_proba(X_v)[:, 1]

        fold_metrics['Accuracy'].append(accuracy_scratch(y_v,  y_pred))
        fold_metrics['Precision'].append(precision_scratch(y_v, y_pred))
        fold_metrics['Recall'].append(recall_scratch(y_v,    y_pred))
        fold_metrics['F1'].append(f1_scratch(y_v,            y_pred))
        fold_metrics['AUC-ROC'].append(roc_auc_scratch(y_v, y_prob)[0])

    print(f"\n{'='*50}")
    print(f"  {label} — {n_splits}-Fold StratifiedKFold Results")
    print(f"{'='*50}")
    rows = []
    for m, values in fold_metrics.items():
        mean = np.mean(values)
        std  = np.std(values)
        rows.append({'Metric': m, 'Mean': mean, 'Std': std,
                     'Min': min(values), 'Max': max(values),
                     'Per-Fold Values': [round(v,3) for v in values]})
        print(f"  {m:<12}: {mean:.4f} ± {std:.4f}  (min={min(values):.3f}, max={max(values):.3f})")
    return pd.DataFrame(rows), fold_metrics

df_bal, fm_bal = evaluate_kfold(X_bal, y_bal, n_splits=5, label='Balanced')
df_imb, fm_imb = evaluate_kfold(X_imb, y_imb, n_splits=5, label='Imbalanced (10% positive)')

## Part 7 — Metric Variance Visualisation

In [ ]:
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
n_folds = 5

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, fm, title in [
    (axes[0], fm_bal, 'Balanced Dataset'),
    (axes[1], fm_imb, 'Imbalanced Dataset (10% pos)')
]:
    x = np.arange(len(metrics_list))
    means = [np.mean(fm[m]) for m in metrics_list]
    stds  = [np.std(fm[m])  for m in metrics_list]

    ax.bar(x, means, color='steelblue', alpha=0.7, capsize=5,
           yerr=stds, error_kw={'ecolor': 'black', 'linewidth': 1.5})
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_list, rotation=30, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

    # Annotate with mean ± std
    for xi, (m, s) in enumerate(zip(means, stds)):
        ax.text(xi, m + s + 0.02, f"{m:.2f}\n±{s:.3f}", ha='center', fontsize=8)

plt.suptitle('K-Fold Metric Comparison: Balanced vs Imbalanced', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 8 — Per-Fold Metric Heatmap

In [ ]:
import matplotlib.colors as mcolors

def plot_fold_heatmap(fold_metrics, title):
    data = np.array([fold_metrics[m] for m in metrics_list])   # (5 metrics × 5 folds)
    fig, ax = plt.subplots(figsize=(9, 4))
    im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=0.0, vmax=1.0)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xticks(range(n_folds))
    ax.set_xticklabels([f"Fold {i+1}" for i in range(n_folds)])
    ax.set_yticks(range(len(metrics_list)))
    ax.set_yticklabels(metrics_list)
    ax.set_title(title)
    for i in range(len(metrics_list)):
        for j in range(n_folds):
            ax.text(j, i, f"{data[i,j]:.3f}", ha='center', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

plot_fold_heatmap(fm_bal, 'Balanced Dataset — Per-Fold Metrics')
plot_fold_heatmap(fm_imb, 'Imbalanced Dataset — Per-Fold Metrics')

# WHY inspect per-fold?  If one fold has dramatically different metrics,
# you likely have a data leakage or distribution shift problem.
# DEBUG TIP: High variance across folds → your model is unstable.
#             Possible solutions: more data, stronger regularisation, simpler model.

## Part 9 — The Accuracy Trap on Imbalanced Data

In [ ]:
# A naive classifier that always predicts '0' (majority class)
# achieves 90% ACCURACY on our imbalanced dataset — but is completely useless.

y_all_neg = np.zeros_like(y_imb[:300])
y_true_sample = y_imb[:300]

naive_acc  = accuracy_scratch(y_true_sample,  y_all_neg)
naive_f1   = f1_scratch(y_true_sample,        y_all_neg)
naive_rec  = recall_scratch(y_true_sample,    y_all_neg)

print("=== Naive 'Always Predict Negative' Classifier ===")
print(f"  Accuracy : {naive_acc:.4f}   ← looks great!")
print(f"  Recall   : {naive_rec:.4f}   ← misses ALL positive cases!")
print(f"  F1       : {naive_f1:.4f}   ← reveals the real picture")
print()
print("Key lesson: On imbalanced data, ALWAYS report F1, Precision, Recall")
print("            alongside Accuracy. Accuracy alone is misleading.")

## Summary

| Metric | When to use | Danger zone |
|---|---|---|
| Accuracy | Balanced classes | Imbalanced data — misleading |
| Precision | Low false positive cost | Miss many real positives |
| Recall | Low false negative cost | Many false alarms |
| F1 | Imbalanced, single metric needed | Can't distinguish P/R tradeoff |
| AUC-ROC | Threshold-independent ranking | Binary classification only |

**K-Fold metric reporting format:**
```
F1: 0.8720 ± 0.0183
     mean    std
```
The std tells you model stability. High std = model is sensitive to which data it trained on.

**Next:** Activity 14 — Final End-to-End ML System Project